# Lexical Analyzer for a Programming Language Using Python

Compiler Design Course Project

Student Name: Apurba_mohajan

Course: compiler design

In [8]:
keywords = {
    "if", "else", "switch", "case", "for", "while", "do",
    "break", "continue", "return",
    "int", "float", "double", "char", "class", "struct",
    "enum", "typedef", "using",
    "public", "private", "protected", "static", "const",
    "volatile", "mutable",
    "async", "await", "throw", "try", "catch", "finally",
    "import", "export", "module", "include", "require",
    "true", "false", "print"
}

In [9]:
import re

token_specification = [
    ("LIBRARY",      r'#include\s*[<"]([\w./]+)[>"]'),
    ("COMMENT",      r'//[^\n]*|/\*[\s\S]*?\*/'),
    ("STRING",       r'"[^"]*"'),
    ("INVALID_NUM",  r'\d+(\.\d*){2,}|\d*(\.\d+){2,}|\d+\.\.+\d*'),
    ("FLOAT",        r'\d+\.\d+([eE][+-]?\d+)?'),
    ("INTEGER",      r'\d+'),
    ("LOGICAL_OP",   r'&&|\|\||!'),
    ("ASSIGN",       r'\+=|-=|\*=|/=|%=|='),
    ("REL_OP",       r'==|!=|<=|>=|<|>'),
    ("ARITH_OP",     r'\+\+|--|\+|-|\*|/|%'),
    ("IDENTIFIER",   r'[a-zA-Z_][a-zA-Z0-9_]*'),
    ("SYMBOL",       r'[()\[\]{};,:]'),
    ("NEWLINE",      r'\n'),
    ("SKIP",         r'[ \t]+'),
    ("MISMATCH",     r'.'),
]

master_pattern = "|".join(
    f"(?P<{name}>{pattern})"
    for name, pattern in token_specification
)

In [10]:
with open("input.txt", "r") as f:
    text = f.read()

In [11]:
tokens = []
line_num = 1

for match in re.finditer(master_pattern, text):
    kind  = match.lastgroup
    value = match.group()

    if kind == "NEWLINE":
        line_num += 1
        continue
    elif kind == "SKIP":
        continue

    elif kind == "LIBRARY":
        lib_name = match.group(1)
        tokens.append(("Library File", lib_name, line_num))
        continue

    elif kind == "COMMENT":
        token_type = "Comment"
        tokens.append((token_type, value, line_num))
        line_num += value.count("\n")
        continue

    elif kind == "INVALID_NUM":
        token_type = "Error: Invalid Number (multiple dots)"
        tokens.append((token_type, value, line_num))
        continue

    elif kind == "IDENTIFIER":
        if value in keywords:
            token_type = "Keyword"
        else:
            rest = text[match.end():]
            ahead = rest.lstrip(" \t")
            if ahead.startswith("("):
                paren = re.match(r'\([^)]*\)', ahead)
                func_call = value + (paren.group() if paren else "()")
                tokens.append(("Function", func_call, line_num))
                continue
            else:
                token_type = "Identifier"

    elif kind == "INTEGER":
        token_type = "Number"
    elif kind == "FLOAT":
        token_type = "Number"
    elif kind == "STRING":
        token_type = "String Literal"
    elif kind == "LOGICAL_OP":
        token_type = "Logical Operator"
    elif kind == "REL_OP":
        token_type = "Relational Operator"
    elif kind == "ASSIGN":
        token_type = "Assignment Operator"
    elif kind == "ARITH_OP":
        token_type = "Arithmetic Operator"
    elif kind == "SYMBOL":
        token_type = "Symbol"
    else:
        token_type = "Unknown"

    tokens.append((token_type, value, line_num))


In [12]:
with open("output.txt", "w") as f:
    for token_type, value, line in tokens:
        f.write(f"{token_type}: {value} (line:{line})\n")

print("Lexical analysis complete. Check output.txt")

Lexical analysis complete. Check output.txt


## CFG for Arithmetic Statement Grammar

```
program      → statement*
statement    → declaration | assignment | print_stmt
declaration  → TYPE IDENTIFIER = expr
assignment   → IDENTIFIER = expr
print_stmt   → print IDENTIFIER
expr         → term (('+' | '-') term)*
term         → factor (('*' | '/') factor)*
factor       → NUMBER | IDENTIFIER | '(' expr ')'
```

In [13]:
class Parser:
    def __init__(self, tokens):
        self.tokens = [t for t in tokens if t[0] not in ("Comment", "Library File")]
        self.pos = 0
        self.symbol_table = {}
        self.errors = []
        self.output = []

    def current(self):
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return ("EOF", "", -1)

    def consume(self, expected_type=None, expected_value=None):
        tok = self.current()
        if expected_type and tok[0] != expected_type:
            self.errors.append(f"Syntax error found in line {tok[2]}: expected {expected_type} but got '{tok[1]}'")
            return None
        if expected_value and tok[1] != expected_value:
            self.errors.append(f"Syntax error found in line {tok[2]}: expected '{expected_value}' but got '{tok[1]}'")
            return None
        self.pos += 1
        return tok

    def parse(self):
        while self.current()[0] != "EOF":
            tok = self.current()
            if tok[0] == "Error: Invalid Number (multiple dots)":
                self.errors.append(f"Syntax error found in line {tok[2]}: invalid number '{tok[1]}'")
                self.pos += 1
                self.skip_to_next_statement()
            elif tok[0] == "Keyword" and tok[1] in ("int", "float", "double"):
                self.parse_declaration()
            elif tok[0] == "Keyword" and tok[1] == "print":
                self.parse_print()
            elif tok[0] == "Identifier":
                self.parse_assignment()
            else:
                self.errors.append(f"Syntax error found in line {tok[2]}: unexpected token '{tok[1]}'")
                self.pos += 1
                self.skip_to_next_statement()

    def skip_to_next_statement(self):
        while self.current()[0] not in ("EOF",) and self.current()[0] != "Symbol" or \
              (self.current()[0] == "Symbol" and self.current()[1] != ";"):
            if self.current()[0] == "EOF":
                break
            if self.current()[0] == "Keyword" and self.current()[1] in ("int", "float", "double", "print"):
                break
            if self.current()[0] == "Identifier" and self.pos > 0:
                prev = self.tokens[self.pos - 1]
                if prev[0] == "Keyword":
                    break
            self.pos += 1
        if self.current()[0] == "Symbol" and self.current()[1] == ";":
            self.pos += 1

    def parse_declaration(self):
        type_tok = self.consume("Keyword")
        name_tok = self.consume("Identifier")
        if name_tok is None:
            self.skip_to_next_statement()
            return
        assign_tok = self.consume("Assignment Operator", "=")
        if assign_tok is None:
            self.skip_to_next_statement()
            return
        val = self.parse_expr()
        if val is not None:
            self.symbol_table[name_tok[1]] = {"type": type_tok[1], "value": val}
        if self.current()[0] == "Symbol" and self.current()[1] == ";":
            self.pos += 1

    def parse_assignment(self):
        name_tok = self.consume("Identifier")
        assign_tok = self.consume("Assignment Operator", "=")
        if assign_tok is None:
            self.skip_to_next_statement()
            return
        val = self.parse_expr()
        if val is not None:
            dtype = "float" if isinstance(val, float) else "int"
            self.symbol_table[name_tok[1]] = {"type": dtype, "value": val}
        if self.current()[0] == "Symbol" and self.current()[1] == ";":
            self.pos += 1

    def parse_print(self):
        self.consume("Keyword", "print")
        name_tok = self.consume("Identifier")
        if name_tok is None:
            return
        if name_tok[1] in self.symbol_table:
            self.output.append(str(self.symbol_table[name_tok[1]]["value"]))
        else:
            self.errors.append(f"Syntax error found in line {name_tok[2]}: undefined variable '{name_tok[1]}'")
        if self.current()[0] == "Symbol" and self.current()[1] == ";":
            self.pos += 1

    def parse_expr(self):
        left = self.parse_term()
        while self.current()[0] == "Arithmetic Operator" and self.current()[1] in ("+", "-"):
            op = self.consume("Arithmetic Operator")
            right = self.parse_term()
            if left is None or right is None:
                return None
            left = left + right if op[1] == "+" else left - right
        return left

    def parse_term(self):
        left = self.parse_factor()
        while self.current()[0] == "Arithmetic Operator" and self.current()[1] in ("*", "/"):
            op = self.consume("Arithmetic Operator")
            right = self.parse_factor()
            if left is None or right is None:
                return None
            if op[1] == "/" and right == 0:
                self.errors.append(f"Syntax error found in line {self.current()[2]}: division by zero")
                return None
            left = left * right if op[1] == "*" else left / right
        return left

    def parse_factor(self):
        tok = self.current()
        if tok[0] == "Number":
            self.pos += 1
            return float(tok[1]) if "." in tok[1] else int(tok[1])
        elif tok[0] == "Identifier":
            self.pos += 1
            if tok[1] in self.symbol_table:
                return self.symbol_table[tok[1]]["value"]
            else:
                self.errors.append(f"Syntax error found in line {tok[2]}: undefined variable '{tok[1]}'")
                return None
        elif tok[0] == "Symbol" and tok[1] == "(":
            self.pos += 1
            val = self.parse_expr()
            if self.current()[0] == "Symbol" and self.current()[1] == ")":
                self.pos += 1
            else:
                self.errors.append(f"Syntax error found in line {tok[2]}: missing closing ')'")
            return val
        elif tok[0] == "Error: Invalid Number (multiple dots)":
            self.errors.append(f"Syntax error found in line {tok[2]}: invalid number '{tok[1]}'")
            self.pos += 1
            return None
        else:
            self.errors.append(f"Syntax error found in line {tok[2]}: unexpected token '{tok[1]}'")
            return None

    def print_symbol_table(self):
        print("\n--- Symbol Table ---")
        print(f"{'Name':<15} {'Type':<10} {'Value'}")
        print("-" * 35)
        for name, info in self.symbol_table.items():
            print(f"{name:<15} {info['type']:<10} {info['value']}")

print("Parser class defined.")

Parser class defined.


In [14]:
parser = Parser(tokens)
parser.parse()

print("=== Errors ===")
if parser.errors:
    for e in parser.errors:
        print(e)
else:
    print("No errors found.")

print("\n=== Output ===")
if parser.output:
    for line in parser.output:
        print(line)
else:
    print("No print statements executed.")

parser.print_symbol_table()

=== Errors ===
No errors found.

=== Output ===
4.0

--- Symbol Table ---
Name            Type       Value
-----------------------------------
a               int        4.0
